# 03 Hybrid retrieval and reranking

## Learning objectives

- compare text, vector, hybrid, and hybrid-plus-reranker configurations;
- explain Reciprocal Rank Fusion without comparing incompatible raw scores;
- keep candidate generation separate from the final context size;
- turn on Azure semantic ranking only through an explicit provider option.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()

## One fixed dataset, four retrieval configurations

Text retrieval is precise for error codes and product names. Vector retrieval
helps with symptoms and paraphrases. Azure AI Search and Databricks AI Search
both offer managed hybrid retrieval; the exact implementation and score ranges
remain provider-native. The useful question is which configuration improves
row-level outcomes on this application's cases.


In [ ]:
from agentic_ops_rag import RetrievalMode
from agentic_ops_rag.evaluation import benchmark, load_cases

pipeline = session.offline_pipeline()
cases = load_cases(course_root / "data" / "evaluation_cases.jsonl")
retrieval_matrix = {
    "A_text": benchmark(pipeline, cases, mode=RetrievalMode.TEXT),
    "B_vector": benchmark(pipeline, cases, mode=RetrievalMode.VECTOR),
    "C_hybrid": benchmark(pipeline, cases, mode=RetrievalMode.HYBRID),
    "D_hybrid_reranked": benchmark(
        pipeline,
        cases,
        mode=RetrievalMode.HYBRID,
        semantic_rerank=True,
    ),
}
retrieval_matrix

All latency values are labelled `simulated_offline_fixture`. They make the
shape of a trade-off visible but are not an SLA estimate. Inspect individual
cases before choosing a winner: an average can hide an exact-code regression,
an authorization failure, or lost answerable coverage.


In [ ]:
from agentic_ops_rag.offline import reciprocal_rank_fusion

text_ranking = ["runbook-exact-code", "runbook-general", "runbook-symptoms"]
vector_ranking = ["runbook-symptoms", "runbook-general", "runbook-exact-code"]
rrf_scores = reciprocal_rank_fusion((text_ranking, vector_ranking))
sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)

RRF combines rank positions, not BM25 and cosine magnitudes. Azure semantic
ranking happens after hybrid fusion and emits a separate reranker score. Never
copy one absolute threshold across BM25, vector, RRF, semantic ranker, and a
different search provider.


In [ ]:
# YOUR TURN — TODO: choose candidate and context counts, then state the budget.
candidate_k = 50
context_k = 8
latency_budget_ms = 750
assert context_k < candidate_k

In [ ]:
# CHECK YOUR WORK
assert candidate_k == 50, "Semantic ranker needs a broad candidate set to test"
assert 1 <= context_k <= 10, "Keep the final model context deliberately bounded"
assert latency_budget_ms > 0
"Candidate generation and final context are separate decisions."

In [ ]:
# Reference solution
reference_query_plan = {
    "mode": "hybrid",
    "candidate_k": 50,
    "context_k": 8,
    "filter_mode": "preFilter",
    "semantic_configuration": "operations-semantic",
}
assert reference_query_plan["context_k"] < reference_query_plan["candidate_k"]
reference_query_plan

## Azure AI Search connected query

The current adapter accepts one `top_k`, so this explicit advanced call asks for
50 candidates and slices the final context in application code. That makes the
candidate/context distinction visible rather than silently pretending the SDK
has two knobs. `preFilter` protects tenant scope before vector ranking. The
stable workshop path uses classic hybrid retrieval; agentic retrieval remains
an optional platform-reviewed extension while parts of it are preview.


In [ ]:
RUN_CONNECTED = False
azure_context = None
if RUN_CONNECTED:
    from agentic_ops_rag import authorized_search

    resources = session.connected_components(allow_network=True)
    candidates = authorized_search(
        resources["retriever"],
        "Checkout went down after a deployment",
        tenant_id="tenant-alpha",
        region="eastus",
        allowed_groups=("ops-payments",),
        mode="hybrid",
        top_k=candidate_k,
        provider_options={
            "query_type": "semantic",
            "semantic_configuration_name": "operations-semantic",
            "vector_filter_mode": "preFilter",
        },
    )
    azure_context = candidates[:context_k]
azure_context

Switching to `aai-platform.databricks-search.example.yml` keeps
`operations-knowledge` unchanged. Provider-specific reranker options are an
explicit escape hatch and should be evaluated as separate changes. The
application compares outcomes and trace evidence, never raw scores across
providers. A positive provider score only orders candidates; it cannot make an
unrelated result answerable. The deterministic shell requires identifier or
query/evidence support and abstains when that support is uncertain.


## Knowledge check

Answer from the evidence you produced, not from memory:

1. Why does RRF use ranks instead of adding BM25 and vector scores?
2. Why are candidate_k and context_k separate?
3. Which filter must be applied before retrieval and why?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You ran a four-configuration ablation, inspected RRF arithmetic, and wrote a
connected semantic-query plan with pre-filtered access scope. Lesson 04 turns
normalized documents into MLflow traces, deterministic gates, and optional RAG
judges.
